# INF1032 · G5 — Redação do ENEM 2024 × perfil socioeconômico

**Pergunta do projeto:** quanto o perfil socioeconômico do participante explica a nota da redação
do ENEM 2024?

**Alvo:** `y_nota_ok` — a nota das redações corrigidas **sem problema de status**. Os zeros por fuga
ao tema, texto em branco ou anulação ficam de fora: eles não medem desempenho na escrita, e sim um
problema de forma.

Este é o notebook único do projeto. Ele substitui os doze notebooks numerados: a ordem das seções
**é** a ordem de execução, e cada seção lê o que a anterior deixou pronto.

| Seção | O que faz |
|---|---|
| — | **Objetivo, métrica e critério de sucesso** (as regras do projeto, antes de qualquer resultado) |
| 0 | Setup: caminhos, semente, imports |
| 1 | Carrega os CSVs brutos e junta participantes + redação |
| 2 | Análise exploratória: nota × perfil |
| 3 | Engenharia de features (linha a linha) |
| 4 | Split treino/teste e gravação da base de modelagem |
| 5 | Base de modelagem: leitura e amostra de treino |
| 6 | **Pipeline de preparo** (`preparo.py` + `ColumnTransformer`) |
| 7 | Modelo de referência validado (regressão linear, CV de 5 partes) |
| 8 | Diagnóstico do linear: vazamento, coeficientes, resíduos, equidade |
| 9 | Modelos candidatos (Ridge, Random Forest, boosting) |
| 10 | Tuning de hiperparâmetros |
| 11 | Ensemble (Voting e Stacking) |
| 12 | Interpretabilidade (permutation importance e SHAP) |
| 13 | Avaliação no conjunto de **teste** — uma vez só |
| 14 | Checklist ético e de viés |
| 15 | Deploy (Streamlit) |

⚠️ **O conjunto de teste fica guardado até a seção 13.** Tudo entre as seções 5 e 12 roda só no
treino: se olharmos o teste antes de decidir o modelo, ele deixa de ser uma medida honesta.

## Objetivo, métrica e critério de sucesso

Esta seção fixa as regras do projeto **antes** de qualquer resultado. Se uma decisão daqui mudar,
as seções 5 em diante precisam rodar de novo.

### Pergunta

**Quanto o perfil socioeconômico do participante explica a nota da redação do ENEM 2024 — e para
quem o modelo erra mais?**

### Para que serve (e para que não serve)

O resultado serve para **direcionar apoio**: mostrar a escolas, secretarias de educação e projetos
de reforço (ex.: oficinas de redação) que grupos de estudantes tendem a precisar de mais apoio, e
quanto do desempenho na escrita está ligado a condições fora do controle do aluno.

Ele **não** serve para selecionar, ranquear ou restringir ninguém. O modelo descreve associações
em uma população; ele não diz a nota que uma pessoa específica "merece" nem explica por que ela
tirou essa nota (associação não é causa).

### Dados e alvo

- **Fonte:** microdados do ENEM 2024 (INEP) — `PARTICIPANTES_2024` × `COMPETENCIAS_REDACAO_2024`,
  4.332.944 inscritos.
- **Alvo:** `y_nota_ok`, a nota final das redações **corrigidas sem problema de status**.
  Ficam fora (i) quem não fez a redação — não existe nota para prever, e imputar inventaria o alvo;
  (ii) as redações zeradas por status (em branco, fuga ao tema, anulada), porque o zero mede um
  problema de forma, não o desempenho na escrita.
- **Tipo de problema:** regressão supervisionada.

### Métrica

| | Métrica | Por quê |
|---|---|---|
| **Principal** | **MAE** (erro absoluto médio), em pontos de nota | É o erro típico de uma pessoa, na mesma unidade da nota ("o modelo erra, em média, X pontos"). Pesa todos os erros igualmente: os erros muito grandes, em parte, vêm da própria discordância entre corretores, e não queremos que eles decidam a escolha do modelo. |
| Apoio | RMSE, em pontos | Mostra se há erros grandes concentrados (se o RMSE cresce muito mais que o MAE). |
| Apoio | R² | Responde direto a pergunta: quanto da variação da nota o perfil explica. |

Toda métrica é reportada como **média ± desvio das 5 rodadas da validação cruzada** (seção 7).
O tuning (seção 10) e a escolha do modelo (seção 12.1) usam o **MAE**.

> **Mudança em relação ao Plano v1 (G1):** a métrica principal era o RMSE. Trocamos para o MAE
> porque o alvo tem um ruído grande e conhecido — os dois corretores divergem, em média, 87 pontos
> (seção 2.10) — e o RMSE dá peso extra justamente a esses erros. O RMSE continua na tabela.

### Critério de sucesso

1. **Ganhar da referência de verdade.** Um modelo só substitui a regressão linear (seção 7) se o
   MAE dele for menor **por mais que as barras de erro** (soma dos dois desvios). Empate → fica a
   linear, que é mais barata e tem coeficientes legíveis.
2. **"Bom o suficiente":** MAE abaixo de **113 pontos** na validação cruzada. É o mesmo patamar do
   Plano v1 (R² acima de 0,27), traduzido para MAE.
3. **Teto realista.** Esperamos R² moderado (em torno de 0,30). A discordância entre corretores
   limita quanto qualquer modelo consegue acertar; um R² muito alto é sinal de vazamento, não de
   acerto.
4. **Erro comparável entre grupos.** O MAE é calculado por tipo de escola, sexo, região e
   cor/raça (seções 8.5 e 14). Um grupo com MAE mais de 10% acima do geral, ou com resíduo médio
   longe de zero (o modelo subestima ou superestima sistematicamente aquele grupo), vira um
   **limite declarado** na conclusão.
5. **Teste uma vez só.** O conjunto de teste (20%, separado na seção 4) só é usado na seção 13,
   depois do modelo escolhido. O número que se reporta é o do teste.

### Atributo sensível: cor/raça

`cor_raca` **não entra como feature** em nenhum modelo. Ela é usada **só na auditoria de
equidade** (erro por grupo). Motivo: um modelo que usa cor/raça para prever a nota pode reproduzir
desigualdades como se fossem "características" das pessoas — exatamente o oposto do uso que
queremos.

### O que ficou de fora — e por quê

| Ficou de fora | Por quê |
|---|---|
| Risco de zerar a redação (classificação, notebook 07) | É outra pergunta (forma, não desempenho). Vira uma linha nos limites. |
| `y_alto`, `y_fez_redacao`, `y_nota` como alvos | Um alvo só, para o projeto ter uma resposta. |
| Clusterização (notebook 05) | Não responde à pergunta do projeto. |
| Target encoding de UF e município | Só aparece na seção 8.1, como **demonstração de vazamento**. |
| Classes do Sisu como alvo | O Sisu entra só para **comunicar** o resultado: a nota prevista vira Baixa / Intermediária / Competitiva. Não é outro modelo. |

## 0 · Setup

Caminhos, semente e opções de exibição, num lugar só. Todos os caminhos são **relativos à pasta do
notebook** — no Colab, `/content`:

- `dados_brutos/` recebe os CSVs do INEP (a seção 1 baixa sozinha se eles não estiverem lá);
- `data_processed/` recebe os derivados do projeto.

Para usar outra pasta, defina as variáveis de ambiente `ENEM_DIR` e `ENEM_OUT` antes de rodar.

O notebook importa o `preparo.py` (seção 6). Se ele não estiver ao lado do notebook — num Colab
novo, por exemplo —, a célula abaixo baixa a versão do repositório do grupo no GitHub.

As seções 1 a 4 são caras e só precisam rodar uma vez: elas gravam
`enem2024_base_bruta.parquet` e `enem2024_features.parquet`. A partir da seção 5 o notebook lê
esses arquivos. Se eles já existirem, dá para pular direto para a seção 5 (rodando antes esta
célula de Setup).

In [ ]:
import gc
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42

DATA_DIR = os.environ.get("ENEM_DIR", "dados_brutos")    # CSVs brutos do INEP (a seção 1 baixa)
OUT_DIR  = os.environ.get("ENEM_OUT", "data_processed")  # derivados do projeto
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

PART_PATH     = os.path.join(DATA_DIR, "PARTICIPANTES_2024.csv")
RED_PATH      = os.path.join(DATA_DIR, "COMPETENCIAS_REDACAO_2024.csv")
BASE_PATH     = os.path.join(OUT_DIR, "enem2024_base_bruta.parquet")   # saída da seção 1
FEATURES_PATH = os.path.join(OUT_DIR, "enem2024_features.parquet")     # saída da seção 4

# colunas dos CSVs brutos
PERFIL   = ["TP_FAIXA_ETARIA","TP_SEXO","TP_ESTADO_CIVIL","TP_COR_RACA","TP_NACIONALIDADE",
            "TP_ST_CONCLUSAO","TP_ANO_CONCLUIU","TP_ENSINO","IN_TREINEIRO","SG_UF_PROVA"]
QUESTOES = [f"Q{i:03d}" for i in range(1, 24)]
CATEG    = PERFIL + QUESTOES

COMP     = [f"NU_NOTA_COMP{k}" for k in range(1, 6)]
AV_NOTA  = ["NU_NOTA_AV1", "NU_NOTA_AV2", "NU_NOTA_AV3", "NU_NOTA_AV4"]
COMP_AV  = [f"NU_NOTA_COMP{k}_AV{a}" for k in range(1, 6) for a in (1, 2)]
RED_COLS = ["NU_INSCRICAO", "TP_STATUS_REDACAO", "NU_NOTA_REDACAO"] + COMP + AV_NOTA + COMP_AV

ALVO  = "y_nota_ok"
ALVOS = ["y_nota", "y_nota_ok"]

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25})
AZUL, LARANJA, VERDE, CINZA = "#06406F", "#c2410c", "#15803d", "#9ca3af"

# preparo.py fica ao lado do notebook; num Colab novo ele não existe, então vem do GitHub do grupo
URL_PREPARO = "https://raw.githubusercontent.com/RodrigoCCLeal/inf1032_3wa05/main/preparo.py"
if not os.path.exists("preparo.py"):
    import urllib.request
    urllib.request.urlretrieve(URL_PREPARO, "preparo.py")
    print("preparo.py baixado do GitHub do grupo")

print("ENEM_DIR:", os.path.abspath(DATA_DIR))
print("ENEM_OUT:", os.path.abspath(OUT_DIR))

## 1 · Obter e carregar os dados

**Fonte:** microdados do ENEM 2024, publicados pelo INEP em
[gov.br/inep → Dados abertos → Microdados → Enem](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem).
Usamos dois arquivos da página:

| Arquivo do INEP | O que tem | CSV que usamos |
|---|---|---|
| `microdados_enem_2024.zip` — *Microdados do Enem 2024* (~530 MB) | perfil e questionário socioeconômico de cada inscrito | `PARTICIPANTES_2024.csv` (4,3 mi linhas) |
| `competencias_redacao_enem_2024.zip` — *Complemento: Notas Detalhadas dos Avaliadores da Redação* (~420 MB) | situação da redação, nota final, nota de cada competência e de cada corretor | `COMPETENCIAS_REDACAO_2024.csv` |

Os dicionários de variáveis vêm dentro de cada zip (pasta `DICIONÁRIO`). Os dois arquivos se ligam
pela chave `NU_INSCRICAO`.

### 1.1 · Download

A célula abaixo só baixa o que falta: se os CSVs já estiverem em `dados_brutos/`, ela não faz nada.
Do zip dos microdados extraímos **só** o `PARTICIPANTES_2024.csv` — o `RESULTADOS_2024.csv` (1,7 GB,
notas das provas objetivas) não é usado pelo projeto. Na primeira vez leva alguns minutos e ocupa
~1,8 GB de disco (zips + CSVs).

In [ ]:
import shutil
import urllib.error
import urllib.request
import zipfile

URLS_INEP = {
    "microdados_enem_2024.zip":
        "https://download.inep.gov.br/microdados/microdados_enem_2024.zip",
    "competencias_redacao_enem_2024.zip":
        "https://download.inep.gov.br/microdados/competencias_redacao_enem_2024.zip",
}
# CSV de que o projeto precisa -> zip do INEP onde ele está
ORIGEM = {PART_PATH: "microdados_enem_2024.zip",
          RED_PATH:  "competencias_redacao_enem_2024.zip"}

def baixar(url, destino, bloco=8 * 2**20):
    # grava em "destino.part" e só renomeia no fim: um download interrompido não vira um zip quebrado
    pedido = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    try:
        resposta = urllib.request.urlopen(pedido, timeout=60)
    except urllib.error.URLError as e:
        raise RuntimeError(
            f"não consegui baixar {url} ({e.reason}). Baixe o zip pela página do INEP e coloque-o "
            f"em {DATA_DIR}/ — a célula extrai o CSV dele.") from e
    total = int(resposta.headers.get("Content-Length") or 0)
    feito, proximo_aviso = 0, 0
    with resposta, open(destino + ".part", "wb") as saida:
        while bloco_lido := resposta.read(bloco):
            saida.write(bloco_lido)
            feito += len(bloco_lido)
            if feito >= proximo_aviso:
                print(f"  {feito / 1e6:,.0f} MB" + (f" de {total / 1e6:,.0f} MB" if total else ""))
                proximo_aviso += 100 * 2**20
    os.replace(destino + ".part", destino)

def obter(csv):
    if os.path.exists(csv):
        print(f"{os.path.basename(csv)}: já está em {DATA_DIR}/ — nada a baixar")
        return
    zip_nome = ORIGEM[csv]
    zip_path = os.path.join(DATA_DIR, zip_nome)
    if not os.path.exists(zip_path):
        print(f"baixando {zip_nome} do INEP...")
        baixar(URLS_INEP[zip_nome], zip_path)
    # extrai só o CSV que interessa, achando-o pelo nome dentro da estrutura de pastas do zip
    with zipfile.ZipFile(zip_path) as z:
        membro = next(n for n in z.namelist() if n.endswith("/" + os.path.basename(csv)))
        print(f"extraindo {membro}")
        with z.open(membro) as origem, open(csv + ".part", "wb") as saida:
            shutil.copyfileobj(origem, saida, 16 * 2**20)
    os.replace(csv + ".part", csv)

for csv in (PART_PATH, RED_PATH):
    obter(csv)
for csv in (PART_PATH, RED_PATH):
    print(f"{os.path.basename(csv):32} {os.path.getsize(csv) / 1e6:8,.0f} MB")

### 1.2 · Leitura e junção

Lê os dois CSVs brutos **uma vez**, junta participantes + redação por `NU_INSCRICAO` e grava
`enem2024_base_bruta.parquet`. Nenhuma outra seção volta a ler CSV bruto.

`CO_MUNICIPIO_PROVA` entra aqui porque o preparo da seção 6 precisa dele; as notas por avaliador
entram porque a EDA (seção 2) olha a divergência entre corretores. Ler a união das colunas uma vez
sai mais barato do que dois merges.

In [ ]:
part = pd.read_csv(PART_PATH, sep=";", encoding="latin1",
                   usecols=["NU_INSCRICAO", "CO_MUNICIPIO_PROVA"] + CATEG,
                   dtype={c: "category" for c in CATEG})
red  = pd.read_csv(RED_PATH, sep=";", encoding="latin1", usecols=RED_COLS,
                   dtype={c: "float32" for c in RED_COLS if c != "NU_INSCRICAO"})

print("PARTICIPANTES:", part.shape, "| NU_INSCRICAO único:", part["NU_INSCRICAO"].is_unique)
print("REDAÇÃO      :", red.shape, "| NU_INSCRICAO único:", red["NU_INSCRICAO"].is_unique)

df = part.merge(red, on="NU_INSCRICAO", how="outer", validate="one_to_one", indicator=True)
print(df["_merge"].value_counts())
df = df.drop(columns="_merge")
del part, red

print("\nBase unida:", df.shape, f"— {df.memory_usage(deep=True).sum() / 1e9:.2f} GB em memória")
df.head()

In [ ]:
df.to_parquet(BASE_PATH, index=False)
print(BASE_PATH, f"{os.path.getsize(BASE_PATH) / 1e6:.1f} MB")

## 2 · Análise exploratória — redação × perfil do participante

Lê `enem2024_base_bruta.parquet` (seção 1). Olhamos uma variável de cada vez antes de cruzar, e
só depois medimos a força de associação de cada uma com a nota.

### 2.1 · Carregar a base da seção 1

In [ ]:
df = pd.read_parquet(BASE_PATH)
df[CATEG] = df[CATEG].astype("category")
print("base:", df.shape)
df.head()

### 2.2 · Rótulos

Os códigos vêm do dicionário do INEP (`Dicionário_Microdados_Enem_2024.xlsx` e
`Dicionário_Competencias_Redacao_2024.xlsx`). Guardamos só os que usaremos nos gráficos.
**Atenção:** em 2024, `Q006` é *"Você possui renda?"*; a **renda familiar é a `Q007`**.

In [ ]:
ROT = {
    "TP_SEXO": {"F": "Feminino", "M": "Masculino"},
    "TP_COR_RACA": {"0": "Não declarado", "1": "Branca", "2": "Preta", "3": "Parda",
                    "4": "Amarela", "5": "Indígena", "6": "Sem informação"},
    "TP_ST_CONCLUSAO": {"1": "Já concluiu", "2": "Conclui em 2024",
                        "3": "Conclui após 2024", "4": "Não concluiu/não cursa"},
    "IN_TREINEIRO": {"0": "Não", "1": "Treineiro"},
    "TP_FAIXA_ETARIA": {"1": "<17", "2": "17", "3": "18", "4": "19", "5": "20", "6": "21", "7": "22",
                        "8": "23", "9": "24", "10": "25", "11": "26-30", "12": "31-35", "13": "36-40",
                        "14": "41-45", "15": "46-50", "16": "51-55", "17": "56-60", "18": "61-65",
                        "19": "66-70", "20": ">70"},
    # escolaridade do pai (Q001) e da mãe (Q002)
    "Q002": {"A": "Nunca estudou", "B": "Fund. I incompleto", "C": "Fund. I completo",
             "D": "Fund. II completo", "E": "Médio completo", "F": "Superior completo",
             "G": "Pós-graduação", "H": "Não sei"},
    "Q007": {"A": "Nenhuma", "B": "até 1,4k", "C": "1,4–2,1k", "D": "2,1–2,8k", "E": "2,8–3,5k",
             "F": "3,5–4,2k", "G": "4,2–5,6k", "H": "5,6–7,1k", "I": "7,1–8,5k", "J": "8,5–9,9k",
             "K": "9,9–11,3k", "L": "11,3–12,7k", "M": "12,7–14,1k", "N": "14,1–16,9k",
             "O": "16,9–21,2k", "P": "21,2–28,2k", "Q": "> 28,2k"},
    "Q020": {"A": "Sem wi-fi", "B": "Com wi-fi"},
    "Q021": {"A": "Nenhum", "B": "1", "C": "2", "D": "3", "E": "4+"},
    "Q023": {"A": "Só pública", "B": "Pública + privada s/ bolsa", "C": "Pública + privada c/ bolsa",
             "D": "Só privada s/ bolsa", "E": "Só privada c/ bolsa", "F": "Não frequentou"},
    "TP_STATUS_REDACAO": {1: "Sem problemas", 2: "Anulada", 3: "Cópia do texto motivador",
                          4: "Em branco", 6: "Fuga ao tema", 7: "Não atende tipo textual",
                          8: "Texto insuficiente", 9: "Parte desconectada"},
}
ROT["Q001"] = ROT["Q002"]

def ordem_codigos(serie):
    # ordena códigos numéricos como número e letras em ordem alfabética
    cats = [c for c in serie.cat.categories]
    return sorted(cats, key=lambda x: (0, int(x)) if str(x).isdigit() else (1, str(x)))

### 2.3 · Valores ausentes

Aqui ausência **tem significado** — por isso **não imputamos** nesta seção:
- `NU_NOTA_REDACAO` ausente → o participante **não fez a redação** (faltou ou foi eliminado);
- `NU_NOTA_AV3` / `AV4` ausentes → não precisou de 3º corretor / banca;
- `TP_ENSINO` só é preenchido para quem já concluiu o EM.

In [ ]:
aus = (df.isna().mean() * 100).round(1)
aus[aus > 0].sort_values(ascending=False).to_frame("% ausente")

### 2.4 · Quem fez a redação?

Antes de olhar notas: **quem ficou de fora** da base de notas? Se a ausência não for aleatória
(e não é), isso limita o que o modelo aprende.

In [ ]:
df["fez_redacao"] = df["NU_NOTA_REDACAO"].notna()
print(f"Inscritos: {len(df):,}  |  fizeram a redação: {df['fez_redacao'].sum():,} "
      f"({df['fez_redacao'].mean():.1%})")

def taxa(col):
    t = (df.groupby(col, observed=True)["fez_redacao"]
           .agg(inscritos="size", taxa_presenca="mean"))
    t["taxa_presenca"] = (t["taxa_presenca"] * 100).round(1)
    t = t.reindex([c for c in ordem_codigos(df[col]) if c in t.index])
    t.index = t.index.map(lambda x: ROT.get(col, {}).get(x, x))
    return t

for col in ["IN_TREINEIRO", "TP_ST_CONCLUSAO", "TP_SEXO"]:
    display(taxa(col))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

t = taxa("TP_FAIXA_ETARIA")
axes[0].bar(t.index, t["taxa_presenca"], color="#3E6E9E")
axes[0].set_title("Presença na redação por faixa etária (%)")
axes[0].tick_params(axis="x", rotation=60)

t = taxa("Q007")
axes[1].bar(t.index, t["taxa_presenca"], color="#3E6E9E")
axes[1].set_title("Presença na redação por renda familiar (Q007, %)")
axes[1].tick_params(axis="x", rotation=60)

plt.tight_layout(); plt.show()

### 2.5 · Situação da redação

Entre quem fez a prova, nem toda redação foi corrigida normalmente. Status diferente de
*Sem problemas* recebe **nota zero** — isso gera uma massa de zeros que não tem a ver com
"escrever mal". É o motivo de o alvo do projeto ser `y_nota_ok` e não a nota crua.

In [ ]:
fez = df[df["fez_redacao"]]
st = fez["TP_STATUS_REDACAO"].value_counts().rename(index=ROT["TP_STATUS_REDACAO"]).to_frame("participantes")
st["%"] = (st["participantes"] / st["participantes"].sum() * 100).round(2)
display(st)

print("Notas zero:", (fez["NU_NOTA_REDACAO"] == 0).sum(),
      "| delas, com status ≠ 'Sem problemas':",
      ((fez["NU_NOTA_REDACAO"] == 0) & (fez["TP_STATUS_REDACAO"] != 1)).sum())
print("Notas 1000:", (fez["NU_NOTA_REDACAO"] == 1000).sum())

### 2.6 · Distribuição da nota

Olhamos uma variável de cada vez **antes** de cruzar. As notas são múltiplos de 20 (média de dois
corretores que dão 0/40/80/…/200 por competência), então usamos *bins* de 20 pontos.

In [ ]:
ok = fez[fez["TP_STATUS_REDACAO"] == 1]   # redações corrigidas normalmente = população do alvo

display(pd.DataFrame({
    "todas que fizeram": fez["NU_NOTA_REDACAO"].describe(),
    "status sem problemas": ok["NU_NOTA_REDACAO"].describe(),
}).round(1))

bins = np.arange(-10, 1011, 20)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
axes[0].hist(fez["NU_NOTA_REDACAO"], bins=bins, color="#999")
axes[0].set_title("Todas as redações (inclui zeros por status)")
axes[1].hist(ok["NU_NOTA_REDACAO"], bins=bins, color="#3E6E9E")
axes[1].set_title("Só status 'Sem problemas'  →  é este o alvo do projeto")
for ax in axes:
    ax.set_xlabel("Nota da redação"); ax.axvline(ok["NU_NOTA_REDACAO"].median(), ls="--", c="k", lw=1)
axes[0].set_ylabel("Participantes")
plt.tight_layout(); plt.show()

### 2.7 · As cinco competências

C1 norma culta · C2 compreensão da proposta · C3 argumentação · C4 coesão · C5 proposta de intervenção.

Vemos (a) como cada competência se distribui e (b) quanto elas andam juntas — se forem muito
correlacionadas, prever a nota total já captura quase tudo.

In [ ]:
dist = pd.DataFrame({c.replace("NU_NOTA_", ""): ok[c].value_counts(normalize=True).sort_index() * 100
                     for c in COMP}).fillna(0)
dist.index = dist.index.astype(int)

ax = dist.T.plot(kind="bar", stacked=True, figsize=(10, 5), colormap="viridis", width=0.7)
ax.set_ylabel("% dos participantes")
ax.set_title("Distribuição da nota em cada competência (status sem problemas)")
ax.legend(title="nota", bbox_to_anchor=(1.01, 1), loc="upper left", ncol=2, fontsize=8)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

ok[COMP + ["NU_NOTA_REDACAO"]].agg(["mean", "median", "std"]).round(1)

In [ ]:
corr = ok[COMP + ["NU_NOTA_REDACAO"]].corr()
labels = [c.replace("NU_NOTA_", "") for c in corr.columns]

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)), labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center",
                color="white" if corr.iloc[i, j] > 0.7 else "black", fontsize=9)
fig.colorbar(im, ax=ax); ax.set_title("Correlação entre competências")
plt.tight_layout(); plt.show()

### 2.8 · Nota × perfil

Agora o cruzamento que interessa ao modelo. Para cada variável: **alunos, média, mediana, desvio** e
a participação do grupo.

Usamos `ok` — as redações **corrigidas sem problema de status**, que é a população do alvo
(`y_nota_ok`). Os zeros por status ficam de fora pelo motivo da seção 2.5: eles medem um problema de
forma, e misturá-los aqui descreveria uma nota que o modelo não vai prever. O
boxplot é desenhado a partir de quantis por grupo (`ax.bxp`) — mais leve do que passar 3 milhões de
pontos.

In [ ]:
ALVO_EDA = "NU_NOTA_REDACAO"

def resumo(col, base=None, alvo=ALVO_EDA):
    base = ok if base is None else base
    g = base[[col, alvo]].astype({alvo: "float64"}).groupby(col, observed=True)[alvo]
    t = g.agg(alunos="count", media="mean", mediana="median", desvio="std").round(1)
    t["% alunos"] = (t["alunos"] / t["alunos"].sum() * 100).round(1)
    t = t.reindex([c for c in ordem_codigos(base[col]) if c in t.index])
    t.index = t.index.map(lambda x: ROT.get(col, {}).get(x, x))
    return t

def boxplot_por(col, titulo, base=None, alvo=ALVO_EDA, ax=None, rot=0):
    base = ok if base is None else base
    q = base.groupby(col, observed=True)[alvo].quantile([.05, .25, .5, .75, .95]).unstack()
    q = q.reindex([c for c in ordem_codigos(base[col]) if c in q.index])
    stats = [{"label": ROT.get(col, {}).get(k, k), "whislo": r[.05], "q1": r[.25], "med": r[.5],
              "q3": r[.75], "whishi": r[.95], "fliers": []} for k, r in q.iterrows()]
    ax = ax or plt.subplots(figsize=(10, 4.5))[1]
    ax.bxp(stats, showfliers=False, patch_artist=True,
           boxprops=dict(facecolor="#BFD3E6"), medianprops=dict(color="#06406F", lw=2))
    ax.set_title(titulo + "  (bigodes = p5–p95)"); ax.set_ylabel("Nota da redação")
    ax.tick_params(axis="x", rotation=rot)
    return ax

#### Renda familiar (Q007)

In [ ]:
display(resumo("Q007"))
boxplot_por("Q007", "Nota da redação por renda familiar mensal (R$)", rot=60)
plt.tight_layout(); plt.show()

#### Escolaridade da mãe (Q002) e do pai (Q001)

In [ ]:
display(resumo("Q002"))
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharey=True)
boxplot_por("Q002", "Escolaridade da mãe", ax=axes[0], rot=45)
boxplot_por("Q001", "Escolaridade do pai", ax=axes[1], rot=45)
plt.tight_layout(); plt.show()

#### Tipo de escola no Ensino Médio (Q023)

In [ ]:
display(resumo("Q023"))
boxplot_por("Q023", "Nota da redação por tipo de escola no EM", rot=20)
plt.tight_layout(); plt.show()

#### Cor/raça, sexo e situação de conclusão

Diferenças entre grupos aqui **se misturam com renda e escola** (confusão). A EDA mostra a
associação; separar os efeitos é trabalho do modelo.

In [ ]:
for col in ["TP_COR_RACA", "TP_SEXO", "TP_ST_CONCLUSAO", "IN_TREINEIRO"]:
    display(resumo(col))

In [ ]:
# renda × tipo de escola: a diferença pública/privada persiste dentro da mesma faixa de renda?
sub = ok.loc[ok["Q023"].isin(["A", "D"]), ["Q007", "Q023", ALVO_EDA]]
tab = (sub.groupby(["Q007", "Q023"], observed=True)[ALVO_EDA].agg(["mean", "count"])
          .unstack("Q023"))
media = tab["mean"].rename(columns={"A": "Só pública", "D": "Só privada s/ bolsa"})
n = tab["count"].rename(columns={"A": "Só pública", "D": "Só privada s/ bolsa"})
media.index = media.index.map(ROT["Q007"])

ax = media.plot(marker="o", figsize=(10, 4.5), color=["#3E8E7E", "#06406F"])
ax.set_ylabel("Nota média da redação"); ax.set_xlabel("Renda familiar")
ax.set_title("Nota por renda, separada por tipo de escola")
ax.set_xticks(range(len(media)), media.index, rotation=60)
plt.tight_layout(); plt.show()
n.index = media.index
n.T

#### Unidade da Federação (local da prova)

In [ ]:
uf_tab = resumo("SG_UF_PROVA").sort_values("media")
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(uf_tab.index, uf_tab["media"],
       yerr=uf_tab["desvio"] / np.sqrt(uf_tab["alunos"]) * 1.96, color="#3E6E9E")
ax.axhline(ok[ALVO_EDA].mean(), ls="--", c="k", lw=1, label="média nacional")
ax.set_ylim(uf_tab["media"].min() - 40, uf_tab["media"].max() + 20)
ax.set_title("Nota média da redação por UF (barra de erro = IC 95%)"); ax.legend()
plt.tight_layout(); plt.show()
uf_tab.sort_values("media", ascending=False)[["alunos", "media", "mediana", "desvio"]]

### 2.9 · Qual variável explica mais, sozinha?

Para escolher *features*, medimos para cada variável o **η² (eta ao quadrado)**: a fração da
variância da nota explicada pelas médias dos grupos daquela variável (é o R² de um modelo que só usa
aquela variável). Medimos na população do alvo (`ok`). Fazemos o mesmo para a **presença** (`fez_redacao`), que não é alvo do projeto mas
diz quem está faltando da base de notas.

- η² = 0 → a variável não separa as notas;
- η² alto → os grupos têm médias bem diferentes.

> Variáveis altamente correlacionadas entre si (renda, bens da casa, escolaridade dos pais) medem
> quase a mesma coisa — os η² **não se somam**.

In [ ]:
def eta2(base, col, alvo):
    y = base[alvo].astype("float64")
    m = base.groupby(col, observed=True)[alvo].transform("mean").astype("float64").fillna(y.mean())
    return ((m - y.mean()) ** 2).mean() / y.var(ddof=0)

df["fez_redacao_num"] = df["fez_redacao"].astype("float32")
DESCR = {"Q001": "escolaridade pai", "Q002": "escolaridade mãe", "Q003": "ocupação pai",
         "Q004": "ocupação mãe", "Q005": "pessoas na casa", "Q006": "possui renda",
         "Q007": "renda familiar", "Q008": "empregado(a) doméstico(a)", "Q009": "banheiros",
         "Q010": "quartos", "Q011": "carro", "Q012": "moto", "Q013": "geladeira",
         "Q014": "freezer", "Q015": "máq. lavar", "Q016": "micro-ondas", "Q017": "aspirador",
         "Q018": "TV", "Q019": "TV assinatura", "Q020": "wi-fi", "Q021": "computador",
         "Q022": "celulares", "Q023": "tipo de escola",
         "TP_FAIXA_ETARIA": "faixa etária", "TP_SEXO": "sexo", "TP_ESTADO_CIVIL": "estado civil",
         "TP_COR_RACA": "cor/raça", "TP_NACIONALIDADE": "nacionalidade",
         "TP_ST_CONCLUSAO": "situação do EM", "TP_ANO_CONCLUIU": "ano de conclusão",
         "TP_ENSINO": "tipo de ensino", "IN_TREINEIRO": "treineiro", "SG_UF_PROVA": "UF da prova"}

forca = pd.DataFrame({
    "descricao": [DESCR.get(c, c) for c in CATEG],
    "eta2_nota": [eta2(ok, c, ALVO_EDA) for c in CATEG],
    "eta2_presenca": [eta2(df, c, "fez_redacao_num") for c in CATEG],
}, index=CATEG).sort_values("eta2_nota", ascending=False)

fig, ax = plt.subplots(figsize=(9, 8))
lab = [f"{i} · {d}" for i, d in zip(forca.index, forca["descricao"])]
ax.barh(lab[::-1], forca["eta2_nota"][::-1], color="#3E6E9E")
ax.set_xlabel("η² (fração da variância da nota explicada pela variável sozinha)")
ax.set_title("Força de associação com a nota da redação")
plt.tight_layout(); plt.show()
forca.round(3)

### 2.10 · Os corretores: o ruído do próprio alvo

Cada redação tem 2 corretores; se discordarem demais, vai para um 3º e, se persistir, para uma
banca. Isso mostra o **teto do problema**: nenhum modelo vai prever a nota com erro menor que a
discordância entre corretores humanos.

In [ ]:
dif = (ok["NU_NOTA_AV1"] - ok["NU_NOTA_AV2"]).abs()
print(f"|AV1 − AV2|: média {dif.mean():.1f}, mediana {dif.median():.0f}; "
      f"> 100 pontos em {(dif > 100).mean():.1%}")
print(f"Correlação AV1 × AV2 (nota total): {ok['NU_NOTA_AV1'].corr(ok['NU_NOTA_AV2']):.3f}")
print(f"Foram para 3º corretor: {ok['NU_NOTA_AV3'].notna().mean():.1%}"
      f"  |  para banca: {ok['NU_NOTA_AV4'].notna().mean():.1%}")

conc = pd.DataFrame({
    f"C{k}": {"corr AV1×AV2": ok[f"NU_NOTA_COMP{k}_AV1"].corr(ok[f"NU_NOTA_COMP{k}_AV2"]),
              "% notas iguais": (ok[f"NU_NOTA_COMP{k}_AV1"] == ok[f"NU_NOTA_COMP{k}_AV2"]).mean() * 100}
    for k in range(1, 6)}).round(2)
display(conc)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(ok["NU_NOTA_AV1"] - ok["NU_NOTA_AV2"], bins=np.arange(-500, 521, 40), color="#3E6E9E")
ax.set_title("Diferença entre corretor 1 e corretor 2 (status sem problemas)")
ax.set_xlabel("AV1 − AV2"); ax.set_ylabel("Redações")
plt.tight_layout(); plt.show()

### 2.11 · Síntese da EDA

**Base.** O merge `PARTICIPANTES` × `COMPETENCIAS_REDACAO` casa **100%** (4.332.944 inscritos, chave
única nos dois lados). **73,1%** fizeram a redação (3.167.955 notas).

**Quem falta não é aleatório.** A presença cai muito entre quem já concluiu o EM (60,8%) e entre os
mais velhos; treineiros comparecem mais (87,8%). A faixa etária é a variável que mais separa presença
(η² ≈ 0,10). → Um modelo treinado só em quem fez a prova **não representa todos os inscritos**. Isso
volta na seção 14.

**O alvo tem duas partes.** 5,2% das redações têm status problemático (principalmente *em branco* e
*fuga ao tema*) e **todas** recebem zero (165 mil). Entre as corrigidas normalmente, a nota vai de 40
a 1000, média 659 e mediana 640 — e só **12 notas 1000**. → O projeto modela a segunda parte: "qual a
nota, dado que a redação foi corrigida sem problema?" (`y_nota_ok`).

**Competências andam juntas** (correlações 0,6–0,8). A C5 (proposta de intervenção) é a mais dispersa
(desvio 58, ~7% tiram 0 nela).

**Perfil socioeconômico importa — e bastante** (médias entre as redações sem problema de status):
- renda familiar: média sobe de **581** (sem renda) para **~784** (acima de R$ 21 mil);
- escolaridade da mãe: **559** (nunca estudou) → **735** (pós-graduação);
- tipo de escola: **625** (só pública) vs **766** (só privada sem bolsa);
- a diferença pública × privada **persiste dentro da mesma faixa de renda** (de ~65 a ~125 pontos,
  maior nas faixas mais baixas), ou seja, escola não é só proxy de renda;
- mulheres têm média maior (673 vs 637); diferenças por cor/raça (Branca 694, Parda 636, Preta 617,
  Indígena 572) se misturam com renda e escola.

**Força das variáveis (η²).** Nenhuma variável sozinha explica mais de ~12% da variância: tipo de
escola (0,12), renda (0,11), escolaridade da mãe (0,09), ocupação do pai, computador em casa e
escolaridade do pai (≈0,085). Muitas medem a mesma coisa (nível socioeconômico) — cuidado com
redundância, que volta na seção 8.

**O alvo é ruidoso.** Os dois corretores diferem em média **87 pontos**; um terço das redações vai
para 3º corretor. A correlação entre corretores por competência fica entre 0,48 (C1) e 0,78 (C5).
Isso limita o quanto **qualquer** modelo consegue acertar: se um modelo der R² muito alto, a primeira
hipótese é vazamento, não genialidade.

**Implicações para a modelagem**
1. **Alvo:** `y_nota_ok` (regressão) entre as redações corrigidas sem problema de status.
2. **Features:** Q023, Q007, Q001–Q004, Q021, Q009–Q011, sexo, faixa etária, UF — categóricas que a
   seção 3 transforma em número. **Cor/raça não é feature**: fica só para a auditoria de equidade
   (seção Objetivo).
3. **Referência:** regressão linear; os modelos da seção 9 precisam superá-la **por mais que as
   barras de erro**.
4. **Validação:** split treino/teste **estratificado por UF** (seção 4) e CV de 5 partes no treino.

In [ ]:
del fez, ok, df
gc.collect()

## 3 · Engenharia de features

Entrada: `enem2024_base_bruta.parquet` (seção 1). Aqui ficam só as transformações **linha a linha**:
cada valor é calculado com os próprios dados daquela pessoa, sem olhar o resto da base. Por isso elas
podem rodar uma vez e ser gravadas — não há nada para "aprender" e, portanto, nada para vazar.

As transformações que **olham o conjunto** (porte do município, percentis da winsorização, medianas
da imputação) **não ficam aqui**: elas foram para o `preparo.py` e rodam dentro do pipeline, na
seção 6. Ver o cabeçalho daquela seção para o porquê.

### 3.1 · Leitura da base bruta

In [ ]:
df = pd.read_parquet(BASE_PATH)
df[CATEG] = df[CATEG].astype("category")
print("base:", df.shape)
df.head()

### 3.2 · Alvos

O projeto tem **um** alvo de modelagem, `y_nota_ok`. `y_nota` fica junto porque a seção 4 precisa
dela para os target encodings.

In [ ]:
df["y_nota"]    = df["NU_NOTA_REDACAO"].astype("float32")              # todos que fizeram
df["y_nota_ok"] = df["y_nota"].where(df["TP_STATUS_REDACAO"] == 1)     # só corrigidas sem problema

df.drop(columns=["NU_NOTA_REDACAO", "TP_STATUS_REDACAO"]
                + [f"NU_NOTA_COMP{k}" for k in range(1, 6)]
                + AV_NOTA + COMP_AV, inplace=True)
df[ALVOS].describe().round(3)

### 3.3 · Codificação ordinal do questionário

As respostas do questionário são letras com ordem natural (A < B < C …). Viram número; as opções
"não sei" viram `NaN` **e** uma flag própria, porque "não sei" não é escolaridade média — costuma
indicar ausência do pai/mãe, e é informativo por si só.

In [ ]:
ESCOLARIDADE = {"A":0,"B":1,"C":2,"D":3,"E":4,"F":5,"G":6}          # H = "não sei" -> NaN + flag
OCUPACAO     = {"A":1,"B":2,"C":3,"D":4,"E":5}                      # F = "não sei" -> NaN + flag
BIN          = {"A":0,"B":1}
QTD_4        = {"A":0,"B":1,"C":2,"D":3}
QTD_5        = {"A":0,"B":1,"C":2,"D":3,"E":4}
# ponto médio da faixa de renda em R$ (última faixa: 1,2x o limite inferior)
RENDA_MEIO   = {"A":0,"B":706,"C":1765,"D":2471,"E":3177,"F":3883,"G":4942,"H":6354,"I":7766,
                "J":9178,"K":10590,"L":12002,"M":13414,"N":15532,"O":19062,"P":24710,"Q":33888}
IDADE_MEIO   = {"1":16,"2":17,"3":18,"4":19,"5":20,"6":21,"7":22,"8":23,"9":24,"10":25,"11":28,
                "12":33,"13":38,"14":43,"15":48,"16":53,"17":58,"18":63,"19":68,"20":73}
ANOS_CONCL   = {"1":1,"2":2,"3":3,"4":4,"5":5,"6":6,"7":7,"8":8,"9":9,"10":10,"11":11,"12":12,
                "13":13,"14":14,"15":15,"16":16,"17":17,"18":19}                # 0 = não informado
REGIAO       = {**{uf:"Norte"        for uf in ["AC","AP","AM","PA","RO","RR","TO"]},
                **{uf:"Nordeste"     for uf in ["AL","BA","CE","MA","PB","PE","PI","RN","SE"]},
                **{uf:"Centro-Oeste" for uf in ["DF","GO","MT","MS"]},
                **{uf:"Sudeste"      for uf in ["ES","MG","RJ","SP"]},
                **{uf:"Sul"          for uf in ["PR","RS","SC"]}}

def mapear(col, dicionario, tipo="float32"):
    return df[col].astype("object").map(dicionario).astype(tipo)

BENS_QTD  = {"Q008":QTD_4,"Q009":QTD_4,"Q010":QTD_4,"Q011":QTD_4,"Q012":QTD_4,"Q013":QTD_4,
             "Q018":QTD_4,"Q021":QTD_5,"Q022":QTD_5}
BENS_BIN  = {"Q014":BIN,"Q015":BIN,"Q016":BIN,"Q017":BIN,"Q019":BIN,"Q020":BIN}

for q, dic in {**BENS_QTD, **BENS_BIN}.items():
    df[f"n_{q.lower()}"] = mapear(q, dic)

df["esc_pai"]  = mapear("Q001", ESCOLARIDADE)
df["esc_mae"]  = mapear("Q002", ESCOLARIDADE)
df["ocup_pai"] = mapear("Q003", OCUPACAO)
df["ocup_mae"] = mapear("Q004", OCUPACAO)
df["pessoas_casa"]   = df["Q005"].astype("object").astype("float32")
df["tem_renda"]      = mapear("Q006", BIN)
df["renda_familiar"] = mapear("Q007", RENDA_MEIO)
df["idade"]          = mapear("TP_FAIXA_ETARIA", IDADE_MEIO)
df["anos_desde_conclusao"] = mapear("TP_ANO_CONCLUIU", ANOS_CONCL)

BENS = [f"n_{q.lower()}" for q in list(BENS_QTD) + list(BENS_BIN)]
df[["esc_pai","esc_mae","ocup_pai","ocup_mae","renda_familiar","pessoas_casa","idade"]].describe().round(2)

### 3.4 · Features derivadas

Razões e flags, todas linha a linha.

Os dois índices por z-score — `indice_bens` (média dos z-scores dos 15 bens) e `inse_proxy` (média
dos z-scores de renda per capita, escolaridade dos pais e `indice_bens`) — **não** ficam aqui. Um
z-score usa a média e o desvio **da base**; calculado aqui, ele usaria também as linhas de teste e as
de validação de cada rodada. Por isso os dois foram para o `preparo.py` (seção 6), que aprende a média
e o desvio só no treino.

In [ ]:
SM_2024 = 1412.0

# --- renda ---
df["renda_pc"]     = (df["renda_familiar"] / df["pessoas_casa"]).astype("float32")
df["log_renda_pc"] = np.log1p(df["renda_pc"]).astype("float32")
df["renda_sm"]     = (df["renda_familiar"] / SM_2024).astype("float32")
df["sem_renda"]    = (df["renda_familiar"] == 0).astype("int8")

# --- escolaridade e ocupação dos pais ---
df["esc_max_pais"]   = df[["esc_pai","esc_mae"]].max(axis=1).astype("float32")
df["esc_media_pais"] = df[["esc_pai","esc_mae"]].mean(axis=1).astype("float32")
df["pais_superior"]  = (df["esc_max_pais"] >= 5).astype("int8")
df["nao_sabe_esc"]   = ((df["Q001"] == "H") | (df["Q002"] == "H")).astype("int8")
df["ocup_max_pais"]  = df[["ocup_pai","ocup_mae"]].max(axis=1).astype("float32")
df["nao_sabe_ocup"]  = ((df["Q003"] == "F") | (df["Q004"] == "F")).astype("int8")

# --- bens: indice_bens é calculado no preparo.py (z-score aprende média e desvio) ---

# --- densidade domiciliar e acesso digital ---
df["comodos_por_pessoa"]    = ((df["n_q009"] + df["n_q010"]) / df["pessoas_casa"]).astype("float32")
df["computador_por_pessoa"] = (df["n_q021"] / df["pessoas_casa"]).astype("float32")
df["celular_por_pessoa"]    = (df["n_q022"] / df["pessoas_casa"]).astype("float32")
df["tem_computador"]        = (df["n_q021"] > 0).astype("int8")
df["acesso_digital"]        = ((df["n_q021"] > 0) & (df["n_q020"] > 0)).astype("int8")

# --- inse_proxy também está no preparo.py, pelo mesmo motivo ---

# --- escola ---
df["escola_publica"]   = (df["Q023"] == "A").astype("int8")
df["escola_privada"]   = df["Q023"].isin(["D","E"]).astype("int8")
df["escola_mista"]     = df["Q023"].isin(["B","C"]).astype("int8")
df["bolsista"]         = df["Q023"].isin(["C","E"]).astype("int8")
df["sem_ensino_medio"] = (df["Q023"] == "F").astype("int8")

# --- trajetória escolar ---
df["treineiro"]   = df["IN_TREINEIRO"].astype("object").astype("int8")
df["concluinte"]  = (df["TP_ST_CONCLUSAO"] == "2").astype("int8")
df["ja_concluiu"] = (df["TP_ST_CONCLUSAO"] == "1").astype("int8")
df["atraso_idade"] = np.where(df["concluinte"] == 1, df["idade"] - 17, np.nan).astype("float32")
df["anos_desde_conclusao"] = df["anos_desde_conclusao"].fillna(0).astype("float32")

# --- demografia ---
df["sexo_fem"] = (df["TP_SEXO"] == "F").astype("int8")
df["cor_raca"] = df["TP_COR_RACA"].astype("object").map(
    {"0":"nao_declarado","1":"branca","2":"preta","3":"parda","4":"amarela",
     "5":"indigena","6":"sem_info"}).astype("category")
df["regiao"]   = df["SG_UF_PROVA"].astype("object").map(REGIAO).astype("category")
df["uf"]       = df["SG_UF_PROVA"].astype("category")

print(df.shape)

## 4 · Split treino/teste e gravação

### 4.1 · Split estratificado por UF

Estratificamos por UF porque a nota média varia bastante entre estados (seção 2.8) e a participação
de cada UF na base é muito desigual — um sorteio simples poderia deixar um estado pequeno
sub-representado no teste.

In [ ]:
rng = np.random.default_rng(SEED)
u = rng.random(len(df)).astype("float32")
corte = pd.Series(u).groupby(df["uf"].to_numpy(), observed=True).quantile(0.8)
df["split"] = np.where(u <= df["uf"].astype("object").map(corte).to_numpy(dtype="float32"),
                       "treino", "teste")
del u
treino = df["split"] == "treino"

pd.crosstab(df["uf"], df["split"], normalize="index").round(3).head()

### 4.2 · Target encodings da UF e do município (fora do pipeline, **de propósito**)

Estes dois encodings substituem o código da UF e do município pela **nota média** daquele lugar,
suavizada. Eles são calculados aqui, **na base de treino inteira** — que é justamente o jeito errado
de fazer.

Mantemos assim de propósito: a seção 8.1 usa esses dois como **demonstração de vazamento**, medindo
quanto de R² eles inflam quando a CV os encontra prontos. Eles **não** entram no modelo de referência
nem em nenhum modelo das seções 9 a 13.

A versão correta do encoding geográfico está no `preparo.py` (`prop_municipio`), ajustada dentro de
cada rodada.

In [ ]:
def target_encode(chave, alvo="y_nota", m=200):
    aux = df.loc[treino & df[alvo].notna(), [chave, alvo]]
    media_geral = aux[alvo].mean()
    g = aux.groupby(chave, observed=True)[alvo].agg(["mean", "count"])
    suave = (g["mean"] * g["count"] + media_geral * m) / (g["count"] + m)
    return df[chave].astype("object").map(suave).fillna(media_geral).astype("float32")

df["te_municipio"] = target_encode("CO_MUNICIPIO_PROVA")
df["te_uf"]        = target_encode("uf")

df.loc[treino, ["te_municipio", "te_uf"]].describe().round(2)

### 4.3 · Matriz de features

Guardamos as colunas que o `preparo.py` exige na entrada (`preparo.ENTRADA`), mais
`CO_MUNICIPIO_PROVA` cru — é ele que vira `prop_municipio` dentro do pipeline.

**Os `NaN` continuam aqui.** A imputação é ajustável (depende da mediana do conjunto) e por isso foi
para o preparo. O parquet guarda o dado como ele é.

In [ ]:
import preparo as prep

COLS_SAIDA = ["NU_INSCRICAO", "split"] + ALVOS + prep.ENTRADA + prep.TE
faltando = [c for c in COLS_SAIDA if c not in df.columns]
assert not faltando, f"faltam colunas para gravar: {faltando}"

# solta as colunas que não vão para o parquet ANTES de reordenar: copiar a base cheia
# (df[COLS_SAIDA].copy()) dobrava a memória e derrubava o kernel com 7 GB
df.drop(columns=[c for c in df.columns if c not in COLS_SAIDA], inplace=True)
gc.collect()
base = df[COLS_SAIDA]
del df
gc.collect()

print("linhas:", len(base), "| colunas gravadas:", base.shape[1])
print("numéricas que o preparo recebe:", len(prep.NUM_ENTRADA),
      "| categóricas:", len(prep.CATEGORICAS))
base[prep.NUM_ENTRADA].isna().mean().round(4).sort_values(ascending=False).head(8).to_frame("% NaN")

### 4.4 · Verificação

In [ ]:
com_alvo = base.loc[base["y_nota_ok"].notna()]
amostra_corr = com_alvo.sample(min(300_000, len(com_alvo)), random_state=SEED)
corr_alvo = (amostra_corr[prep.NUM_ENTRADA].corrwith(amostra_corr["y_nota_ok"])
             .sort_values(key=abs, ascending=False).round(3))
del amostra_corr
display(corr_alvo.head(15).to_frame("corr com y_nota_ok"))

print("treino/teste:", base["split"].value_counts(normalize=True).round(3).to_dict())
print("alvos disponíveis:", {c: int(base[c].notna().sum()) for c in ALVOS})
assert base["NU_INSCRICAO"].is_unique
assert base.loc[base["split"] == "treino", "y_nota_ok"].notna().any()
print("verificações OK")

### 4.5 · Gravação

In [ ]:
def salvar(tabela, nome, chunk=1_000_000):
    # grava em parquet por blocos: evita pico de memória com a base cheia
    caminho = os.path.join(OUT_DIR, nome + ".parquet")
    try:
        import pyarrow as pa, pyarrow.parquet as pq
        writer = None
        for ini in range(0, len(tabela), chunk):
            t = pa.Table.from_pandas(tabela.iloc[ini:ini + chunk], preserve_index=False)
            writer = writer or pq.ParquetWriter(caminho, t.schema, compression="snappy")
            writer.write_table(t)
        writer.close()
    except Exception as e:
        caminho = os.path.join(OUT_DIR, nome + ".csv.gz")
        tabela.to_csv(caminho, index=False, compression="gzip")
        print("parquet indisponível (%s); gravado como csv.gz" % type(e).__name__)
    print(f"{nome}: {tabela.shape} -> {caminho} ({os.path.getsize(caminho)/1e6:.1f} MB)")

salvar(base, "enem2024_features")
# só o que entra no modelo: numéricas do preparo + UF (cor_raca e regiao não são features)
pd.Series(prep.NUMERICAS + prep.CAT_MODELO).to_csv(os.path.join(OUT_DIR, "lista_features.csv"),
                                                  index=False, header=["feature"])
del base
gc.collect()

## 5 · Base de modelagem

A partir daqui o notebook só lê `enem2024_features.parquet`. Se as seções 1 a 4 já rodaram alguma
vez, dá para começar por aqui.

Duas restrições valem até a seção 12:

1. só linhas com `split == "treino"`;
2. só linhas com `y_nota_ok` preenchido — as redações com problema de status ficam de fora, como
   decidido na seção 2.5.

A amostra de `AMOSTRA` linhas deixa a validação cruzada rápida. Com `AMOSTRA = None` roda na base de
treino inteira.

In [ ]:
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (HistGradientBoostingRegressor, RandomForestRegressor,
                              StackingRegressor, VotingRegressor)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.model_selection import (KFold, RandomizedSearchCV, cross_val_predict,
                                     cross_validate, train_test_split, validation_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import preparo as prep
from preparo import preparo

print("scikit-learn", sklearn.__version__)

AMOSTRA = 150_000                                 # None = toda a base de treino
RAPIDO  = True                                    # False = versão que vai para o G2
CV5     = KFold(5, shuffle=True, random_state=0)  # convenção do curso: 5 partes, semente fixa
CV3     = KFold(3, shuffle=True, random_state=0)  # busca interna, mais barata

# MAE é a métrica principal (seção Objetivo); RMSE e R² ficam de apoio
METRICAS = {"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"}

In [ ]:
COLS_LER = ["split"] + ALVOS + prep.ENTRADA + prep.TE
# filters: o parquet entrega só as linhas de treino (o teste nem chega à memória)
base   = pd.read_parquet(FEATURES_PATH, columns=COLS_LER, filters=[("split", "==", "treino")])
treino = base[(base["split"] == "treino") & base[ALVO].notna()]
df_mod = (treino if AMOSTRA is None
          else treino.sample(min(AMOSTRA, len(treino)), random_state=SEED)).reset_index(drop=True)
del base, treino
gc.collect()

X = df_mod[prep.ENTRADA + prep.TE].copy()
y = df_mod[ALVO].astype("float64")

print(f"linhas de treino usadas: {len(X):,}")
print(f"nota: média {y.mean():.1f} | desvio {y.std():.1f} | mín {y.min():.0f} | máx {y.max():.0f}")
X.head(3)

## 6 · Pipeline de preparo

Esta é a peça que faz todo o resto do notebook ser honesto.

### Por que o preparo tem de estar **dentro** do pipeline

Quatro transformações da base olham o **conjunto** de linhas, não cada linha isolada:

| Transformação | O que ela olha |
|---|---|
| porte do município | quantas linhas fizeram prova em cada município |
| winsorização dos quocientes de renda | os percentis 0,5% e 99,5% |
| imputação dos `NaN` | a mediana de cada coluna |
| `indice_bens` e `inse_proxy` (z-scores) | a média e o desvio de cada coluna |

Se qualquer uma delas for calculada **antes** da validação cruzada, as linhas que depois servem de
validação ajudaram a calcular o número que é aplicado a elas mesmas. É o mesmo vazamento que a
seção 8.1 mede nos target encodings — só que silencioso, porque não aparece como coluna nova.

Por isso elas saíram da seção 3 e foram para o `preparo.py`, que é um transformador do
scikit-learn com `fit` e `transform` separados. Dentro de um `Pipeline`, o `cross_validate` chama o
`fit` **só** com as linhas de treino de cada uma das 5 rodadas.

### O que vem depois do preparo

Um `ColumnTransformer` com duas metades:

- **`StandardScaler` nas numéricas.** Na regressão por mínimos quadrados puros padronizar não muda o
  ajuste, mas assim que entram modelos com **penalidade** (Ridge, seção 10) a escala passa a decidir
  o resultado: sem padronizar, o mesmo `alpha` penaliza a renda em reais e a escolaridade em níveis
  com pesos arbitrariamente diferentes. As árvores não se importam com escala, então um
  pré-processamento só serve a todos os modelos do notebook — menos código para manter e comparação
  justa entre eles.
- **`OneHotEncoder(handle_unknown="ignore")` nas categóricas.** O `handle_unknown="ignore"` é o que
  impede o pipeline de quebrar quando uma rodada da CV não tem nenhuma linha de uma categoria rara
  (uma UF pequena): a categoria não vista vira uma linha de zeros em vez de erro.

Só a **UF** entra no one-hot (`prep.CAT_MODELO`). As outras duas categóricas passam pelo preparo mas
**não entram no modelo**:

- `cor_raca` fica **só para a auditoria de equidade** (seções 8.5 e 14), como decidido na seção
  Objetivo. Para medir o erro por grupo não é preciso dar o grupo ao modelo;
- `regiao` é função exata da UF, e as duas juntas dariam colinearidade perfeita.

### A troca do porte do município

A versão antiga era `log_porte_municipio = log1p(contagem de linhas no município)`. O problema é que
**contagem depende do tamanho da amostra**: o mesmo município vale 12.000 na base cheia, 900 numa
amostra de 150 mil e um número diferente em cada rodada da CV, só porque a dobra é menor. O modelo
aprende uma escala que não se reproduz.

A nova é `prop_municipio = value_counts(normalize=True)` — a **proporção** das linhas de treino que
fizeram prova naquele município. A proporção é a mesma grandeza em qualquer tamanho de amostra, e
município que não apareceu no treino daquela rodada recebe 0, que é a leitura correta: nenhuma linha
de treino ali.

In [ ]:
CAT_OHE = prep.CAT_MODELO           # só a UF: cor_raca é só auditoria; regiao é função da UF

def colunas(numericas=None, categoricas=CAT_OHE):
    # StandardScaler nas numéricas, one-hot nas categóricas;
    # handle_unknown="ignore" para a rodada da CV que não vir uma categoria rara
    return ColumnTransformer([
        ("num", StandardScaler(), list(prep.NUMERICAS if numericas is None else numericas)),
        ("cat", OneHotEncoder(handle_unknown="ignore"), list(categoricas)),
    ])

def pipe(modelo, numericas=None, categoricas=CAT_OHE, target_encodings=False):
    # preparo() primeiro: é ele que ajusta porte do município, winsorização e imputação
    # dentro de cada rodada da validação cruzada
    return Pipeline([
        ("preparo", preparo(target_encodings=target_encodings)),
        ("colunas", colunas(numericas, categoricas)),
        ("modelo",  modelo),
    ])

modelo_referencia = pipe(LinearRegression())
modelo_referencia

### 6.1 · Conferindo o preparo

Antes de confiar no pipeline, olhamos o que o preparo faz com um pedaço da base: a proporção do
município, quantas linhas ele marca como outlier de renda e se sobrou algum `NaN`.

In [ ]:
p = preparo().fit(X.iloc[:len(X) // 2])
saida = p.transform(X.iloc[len(X) // 2:])

print(f"municípios vistos no ajuste : {len(p.prop_municipio_):,}")
print(f"soma das proporções         : {p.prop_municipio_.sum():.6f}   (tem de dar 1)")
print(f"maior município             : {p.prop_municipio_.max():.4%} das linhas de treino")
print(f"linhas com município não visto no ajuste -> prop 0 : "
      f"{(saida['prop_municipio'] == 0).sum():,}")
print(f"marcadas como outlier de renda : {int(saida['outlier_renda'].sum()):,} "
      f"({saida['outlier_renda'].mean():.2%})")
print(f"NaN restantes nas numéricas    : {int(saida[prep.NUMERICAS].isna().sum().sum())}")
print(f"colunas entregues              : {saida.shape[1]} "
      f"({len(prep.NUMERICAS)} numéricas + {len(prep.CATEGORICAS)} categóricas)")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(np.log10(p.prop_municipio_.to_numpy()), bins=40, color=AZUL)
ax[0].set_xlabel("log10 da proporção"); ax[0].set_ylabel("municípios")
ax[0].set_title("prop_municipio: distribuição entre municípios")
ax[1].hist(saida["prop_municipio"], bins=40, color=VERDE)
ax[1].set_xlabel("prop_municipio"); ax[1].set_ylabel("participantes")
ax[1].set_title("prop_municipio: distribuição entre pessoas")
plt.tight_layout(); plt.show()
del p, saida

## 7 · Modelo de referência validado

A referência é a **regressão linear**, avaliada com **validação cruzada de 5 partes**, reportando
**média ± desvio**. O piso é o **chute**: prever sempre a média do treino, que por definição dá
R² = 0.

Todo modelo das seções 9 a 11 é comparado com esta tabela, **nas mesmas 5 rodadas** — comparar
modelos em partições diferentes não quer dizer nada.

In [ ]:
def avaliar(nome, modelo, X_=None, y_=None, cv=CV5):
    # roda a CV de 5 partes e devolve média e desvio de cada métrica
    t0 = time.time()
    r = cross_validate(modelo, X if X_ is None else X_, y if y_ is None else y_,
                       cv=cv, scoring=METRICAS, n_jobs=-1)
    linha = {"modelo": nome}
    for m in METRICAS:
        v = r[f"test_{m}"] * (1 if m == "R2" else -1)   # o sklearn devolve erro com sinal negativo
        linha[m], linha[f"{m} dp"] = v.mean(), v.std()
    linha["seg"] = round(time.time() - t0, 1)
    print(f"  {nome:44} {linha['seg']:>7.1f}s")
    return linha

def tabela(linhas):
    t = pd.DataFrame(linhas).set_index("modelo")
    out = pd.DataFrame({m: t[m].map("{:.3f}".format) + " ± " + t[f"{m} dp"].map("{:.3f}".format)
                        for m in METRICAS})
    out["seg"] = t["seg"]
    return out

print("rodando a CV de 5 partes:")
res = [avaliar("Chute: média do treino", DummyRegressor(strategy="mean")),
       avaliar("Regressão linear (referência)", modelo_referencia)]
tabela(res)

**Como ler:** cada célula é **média ± desvio das 5 rodadas**. RMSE e MAE estão em **pontos de nota**.
Um modelo novo só é "melhor" que a referência se a diferença for maior que essas barras de erro —
*um R² sem o desvio padrão é uma opinião*.

> ✍️ **Leitura do grupo** (preencher depois de rodar): o modelo de referência erra, em média, ___
> pontos (MAE) e explica ___ da variação da nota (R²), contra ___ pontos de erro do chute.

## 8 · Diagnóstico do modelo de referência

Quatro perguntas sobre a referência, antes de trocar de modelo: ela está vazando? os coeficientes são
legíveis? o erro é parecido ao longo da nota? e entre grupos de pessoas?

### 8.1 · O vazamento do target encoding

`te_municipio` e `te_uf` foram calculados na seção 4.2 com **todo** o conjunto de treino. Na CV isso
quer dizer que a nota de cada linha de validação **ajudou a calcular** o encoding usado para
prevê-la: o modelo vê um pedaço da resposta.

Comparamos três versões, nas **mesmas** 5 rodadas:

1. **com os TE pré-calculados** (vazam);
2. **sem TE** — a referência da seção 7;
3. **com `prop_municipio` ajustado dentro do pipeline**, que é a referência: já é a versão sem
   vazamento do mesmo sinal geográfico.

In [ ]:
res_te = [avaliar("1 · com te_municipio e te_uf da seção 4 (vazam)",
                  pipe(LinearRegression(), numericas=prep.NUMERICAS + prep.TE,
                       target_encodings=True)),
          res[1] | {"modelo": "2 · sem TE, com prop_municipio (referência)"}]
tabela(res_te)

**Como ler:** se a versão 1 parece melhor que a 2 por **mais** que as barras de erro, parte desse
ganho é otimismo do vazamento, e o número não se repetiria em dados novos. A diferença é o preço de
calcular um encoding fora do pipeline.

> ✍️ **Leitura do grupo:** a diferença entre 1 e 2 foi de ___ no R². Por isso os TE pré-calculados
> não entram em nenhum modelo a partir daqui — o sinal geográfico entra pelo `prop_municipio` e pelo
> one-hot da UF, os dois ajustados dentro de cada rodada.

### 8.2 · Coeficientes com unidade

Com todas as features os coeficientes não são legíveis: há colunas que são combinação exata de
outras (`renda_sm` = `renda_familiar` / 1.412, `esc_media_pais` = média de `esc_pai` e `esc_mae`, as
flags de escola somam 1). O modelo acerta igual, mas "divide o crédito" de forma arbitrária.

Para **ler** os pesos usamos um modelo **enxuto**, com uma coluna por ideia. Primeiro conferimos
quanto de acerto ele perde.

⚠️ **Um detalhe que vem do `StandardScaler`.** Dentro do pipeline o coeficiente está em "pontos de
nota por **desvio padrão** da coluna", não na unidade original. Para ler "pontos por real" é preciso
desfazer a escala: dividir o coeficiente pelo `scale_` que o `StandardScaler` aprendeu. É o que a
função `coeficientes` faz.

In [ ]:
ENXUTO = ["log_renda_pc", "esc_max_pais", "indice_bens", "escola_privada", "escola_mista",
          "bolsista", "sexo_fem", "idade", "treineiro", "acesso_digital"]

enxuto = pipe(LinearRegression(), numericas=ENXUTO)
res_enxuto = [res[1], avaliar("Linear enxuto (10 numéricas + UF)", enxuto)]
tabela(res_enxuto)

In [ ]:
UNIDADE = {
    "log_renda_pc":   "por +1 no log da renda per capita (dobrar a renda = coef × 0,69)",
    "esc_max_pais":   "por nível a mais de escolaridade do pai/mãe mais escolarizado (escala 0–6)",
    "indice_bens":    "por +1 desvio padrão no índice de bens",
    "escola_privada": "privada vs. pública (referência)",
    "escola_mista":   "mista vs. pública (referência)",
    "bolsista":       "bolsista vs. não bolsista",
    "sexo_fem":       "feminino vs. masculino",
    "idade":          "por ano a mais de idade",
    "treineiro":      "treineiro vs. não treineiro",
    "acesso_digital": "tem computador e wi-fi vs. não tem",
}

def coeficientes(pipeline_ajustado, colunas_num):
    # desfaz a padronização: coef por desvio padrão -> coef na unidade original da coluna
    ct     = pipeline_ajustado.named_steps["colunas"]
    escala = pd.Series(ct.named_transformers_["num"].scale_,
                       index=ct.transformers_[0][2])
    coef   = pd.Series(pipeline_ajustado.named_steps["modelo"].coef_,
                       index=ct.get_feature_names_out())
    coef   = coef[[f"num__{c}" for c in colunas_num]].set_axis(colunas_num)
    return coef / escala[colunas_num]

enxuto.fit(X, y)
coef_enxuto = coeficientes(enxuto, ENXUTO)
pd.DataFrame({"coef (pontos)": coef_enxuto.round(1), "leitura": pd.Series(UNIDADE)})

**Como ler um coeficiente:** é a diferença de nota associada à feature **mantendo todas as outras
constantes**. Por exemplo, `escola_privada` compara alunos de privada e de pública **com a mesma
renda, a mesma escolaridade dos pais, a mesma UF**.

⚠️ É **associação, não causa**. O coeficiente não autoriza dizer "mudar o aluno para uma escola
privada aumenta a nota em X pontos": a escola pode estar no lugar de algo que não medimos.

> ✍️ **Leitura do grupo** (escolher dois): mantendo o resto constante, estudar em escola privada está
> associado a ___ pontos a mais que na pública. Dobrar a renda per capita está associado a ___ pontos.

### 8.3 · Multicolinearidade: quando duas colunas dizem a mesma coisa

`inse_proxy` é a média dos z-scores de `log_renda_pc`, `esc_max_pais` e `indice_bens`. Se ele entra
no modelo enxuto, as quatro colunas passam a **dividir o crédito**.

In [ ]:
colunas_inse = ["inse_proxy", "log_renda_pc", "esc_max_pais", "indice_bens"]
Z = preparo().fit_transform(X)   # os índices agora nascem no preparo
print("correlação do inse_proxy com as três colunas que o compõem:")
print(Z[colunas_inse].corr()["inse_proxy"].round(2)[1:].to_string())
del Z

mais = pipe(LinearRegression(), numericas=ENXUTO + ["inse_proxy"]).fit(X, y)
coef_mais = coeficientes(mais, ENXUTO + ["inse_proxy"])
pd.DataFrame({"sem inse_proxy": coef_enxuto, "com inse_proxy": coef_mais}).round(1).loc[
    ["log_renda_pc", "esc_max_pais", "indice_bens", "inse_proxy"]]

**Como ler:** se os coeficientes de renda, escolaridade e bens mudam bastante quando o `inse_proxy`
entra, mas o acerto do modelo quase não muda, é a multicolinearidade: o que fica instável é a
**leitura**, não a previsão. É o argumento para a **regularização** (Ridge, seção 10).

### 8.4 · Resíduos: o erro é parecido ao longo da nota?

Usamos as previsões **fora da amostra** (`cross_val_predict`): cada linha é prevista pela rodada em
que ela estava na validação.

In [ ]:
previsto = cross_val_predict(modelo_referencia, X, y, cv=CV5, n_jobs=-1)
residuo  = y - previsto

print(f"nota real:     {y.min():.0f} a {y.max():.0f}")
print(f"nota prevista: {previsto.min():.0f} a {previsto.max():.0f}")

idx = np.random.default_rng(SEED).choice(len(y), size=min(20_000, len(y)), replace=False)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].scatter(previsto[idx], residuo.iloc[idx], s=3, alpha=.15, color=AZUL)
ax[0].axhline(0, color=LARANJA, lw=1.2)
ax[0].set_xlabel("nota prevista"); ax[0].set_ylabel("resíduo (real − previsto)")
ax[0].set_title("resíduo × previsão")

faixa = pd.cut(y, bins=range(0, 1001, 100), include_lowest=True)
por_faixa = pd.DataFrame({"faixa": faixa, "residuo": residuo}).groupby("faixa", observed=True)["residuo"]
por_faixa.mean().plot(kind="bar", ax=ax[1], color=AZUL)
ax[1].axhline(0, color="#777", lw=1)
ax[1].set_xlabel("faixa da nota REAL"); ax[1].set_ylabel("resíduo médio")
ax[1].set_title("o modelo erra para que lado, em cada faixa")
plt.tight_layout(); plt.show()

pd.DataFrame({"linhas": por_faixa.size(), "resíduo médio": por_faixa.mean().round(1),
              "erro absoluto mediano": por_faixa.apply(lambda r: r.abs().median()).round(1)})

**Como ler:** uma nuvem **sem forma** em volta do zero é o ideal; um **leque** quer dizer que o erro
cresce numa parte da faixa; um **arco** quer dizer que o modelo erra **sistematicamente** para um
lado.

Com um R² moderado, o esperado é que o modelo **puxe as notas para a média**: a faixa de notas
previstas fica bem mais estreita que a real, então ele **subestima** quem tirou nota alta (resíduo
médio positivo) e **superestima** quem tirou nota baixa. Isso é coerente com a pergunta do projeto: o
perfil explica uma parte da nota, e o resto (escrita, preparo, correção) o perfil não vê.

> ✍️ **Leitura do grupo:** a nota prevista vai de ___ a ___, contra ___ a ___ da real. O formato dos
> resíduos é ___, e isso significa que ___.

### 8.5 · Em que faixa o modelo vale, e para quem ele erra mais

Primeiro a **faixa de treino**: fora dela o modelo responde "com a mesma cara", sem avisar
(extrapolação). Depois o **erro por grupo**, que é o primeiro olhar de equidade — a seção 14
consolida.

Resíduo médio **positivo** quer dizer que o modelo **subestima** a nota daquele grupo; **negativo**,
que **superestima**.

In [ ]:
FAIXA = ["renda_pc", "idade", "pessoas_casa", "esc_max_pais"]
Xf = X[FAIXA].astype("float64")
display(pd.DataFrame({"mín": Xf.min(), "p1": Xf.quantile(.01),
                      "p99": Xf.quantile(.99), "máx": Xf.max()}).round(1))

tipo_escola = np.select([X["escola_privada"] == 1, X["escola_mista"] == 1, X["escola_publica"] == 1],
                        ["privada", "mista", "pública"], default="sem EM / outra")

def erro_por_grupo(grupo, res=None):
    res = residuo if res is None else res
    aux = pd.DataFrame({"grupo": np.asarray(grupo), "res": np.asarray(res)})
    g = aux.groupby("grupo")["res"]
    return pd.DataFrame({"linhas": g.size(),
                         "MAE": g.apply(lambda r: r.abs().mean()),
                         "RMSE": g.apply(lambda r: np.sqrt((r ** 2).mean())),
                         "resíduo médio": g.mean()}).round(1).sort_values("linhas", ascending=False)

# os mesmos quatro grupos da seção Objetivo; nenhum deles precisa estar no modelo para ser auditado
display(erro_por_grupo(tipo_escola))
display(erro_por_grupo(np.where(X["sexo_fem"] == 1, "feminino", "masculino")))
display(erro_por_grupo(X["regiao"].astype(str)))
display(erro_por_grupo(X["cor_raca"].astype(str)))

**O modelo vale para:** participantes do **ENEM 2024**, com redação **corrigida sem problema de
status**, com renda, idade e composição familiar **declaradas no questionário** e dentro das faixas
acima. Ele **não** vale para outros anos nem para prever quem zera a redação.

**Como ler o erro por grupo:** um MAE bem maior num grupo quer dizer que o modelo é **menos
confiável** para ele; um resíduo médio longe de zero quer dizer **viés sistemático**. `cor_raca` **não
está no modelo**: ela entra só aqui, para medir o erro por grupo.

Pelo critério da seção Objetivo, um grupo com MAE **mais de 10% acima** do geral, ou com resíduo médio
longe de zero, vira um limite declarado na conclusão.

> ✍️ **Leitura do grupo:** o maior MAE foi em ___ (___ pontos) e o menor em ___ (___). O grupo com
> resíduo médio mais distante de zero foi ___.

## 9 · Modelos candidatos

Todos usam o **mesmo** pipeline da seção 6 (preparo + `StandardScaler` + one-hot) e as **mesmas** 5
rodadas da seção 7. É isso que torna os números comparáveis linha a linha.

**Random Forest** (bagging: muitas árvores fundas em amostras diferentes, média no fim) e
**boosting** (árvores rasas em sequência, cada uma corrigindo o erro da anterior) já **são** métodos
de ensemble. A seção 11 vai além e combina famílias diferentes.

⏱️ O Random Forest é o gargalo. Com `RAPIDO = True` o notebook roda em poucos minutos; com
`RAPIDO = False` roda a versão do G2.

In [ ]:
try:
    from lightgbm import LGBMRegressor
    TEM_LGBM = True
except ImportError:
    TEM_LGBM = False
print("LightGBM disponível:", TEM_LGBM,
      "(se False, o HistGradientBoosting cobre o item do Plano)")

N_ITER = 8 if RAPIDO else 25            # sorteios do RandomizedSearchCV

FLORESTA = RandomForestRegressor(n_estimators=100, min_samples_leaf=50, max_features="sqrt",
                                 random_state=SEED, n_jobs=-1)

candidatos = {
    "Ridge (alpha=1, padrão)": pipe(Ridge(alpha=1.0, random_state=SEED)),
    "Random Forest":           pipe(FLORESTA),
    "Boosting (HistGradient)": pipe(HistGradientBoostingRegressor(random_state=SEED)),
}
if TEM_LGBM:
    candidatos["LightGBM"] = pipe(LGBMRegressor(random_state=SEED, verbose=-1))

print("rodando a CV de 5 partes:")
res += [avaliar(n, m) for n, m in candidatos.items()]
tabela(res)

**Como ler:** um modelo só é melhor que a referência linear se a diferença for **maior que as
barras**. É tentador declarar vencedor por meio ponto de MAE.

Olhe também a coluna `seg`. Se o Random Forest custa 50× o tempo da linear para ganhar 1 ponto de MAE,
essa é uma informação de decisão tão legítima quanto a métrica.

A seção 2 já avisava: **R² moderado é o esperado**, porque o η² de cada variável isolada não passa de
~0,10 e a discordância entre corretores limita o teto.

## 10 · Tuning de hiperparâmetros

### 10.1 · A curva de regularização do Ridge

A seção 8.3 diagnosticou **multicolinearidade**. **Ridge é a resposta a esse diagnóstico**: ele
encolhe os coeficientes e distribui o peso entre colunas correlacionadas em vez de deixar o ajuste
escolher arbitrariamente.

`alpha` é a força do encolhimento. Em vez de chutar, desenhamos a curva inteira. É aqui que o
`StandardScaler` da seção 6 deixa de ser opcional: sem ele, o mesmo `alpha` penalizaria a renda em
reais e a escolaridade em níveis com pesos arbitrariamente diferentes.

In [ ]:
alphas = np.logspace(-2, 5, 24)
tr, te = validation_curve(pipe(Ridge(random_state=SEED)), X, y,
                          param_name="modelo__alpha", param_range=alphas,
                          cv=CV5, scoring="neg_mean_absolute_error", n_jobs=-1)
mae_tr, mae_te = -tr.mean(axis=1), -te.mean(axis=1)
dp_te = te.std(axis=1)
melhor_alpha = alphas[mae_te.argmin()]

fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.semilogx(alphas, mae_tr, "o-", ms=3.5, color=CINZA, label="treino")
ax.semilogx(alphas, mae_te, "o-", ms=3.5, color=AZUL, label="validação (5 rodadas)")
ax.fill_between(alphas, mae_te - dp_te, mae_te + dp_te, color=AZUL, alpha=.15)
ax.axvline(melhor_alpha, ls="--", color=LARANJA, lw=1.3, label=f"melhor alpha = {melhor_alpha:.3g}")
ax.set_xlabel("alpha (força da regularização)"); ax.set_ylabel("MAE (pontos de nota)")
ax.set_title("Ridge: regularização de menos (esquerda) × de mais (direita)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

res.append(avaliar(f"Ridge ajustado (alpha={melhor_alpha:.3g})",
                   pipe(Ridge(alpha=melhor_alpha, random_state=SEED))))
tabela(res)

**Como ler a curva:** à **esquerda** (alpha baixo) o Ridge vira a regressão linear — treino e
validação quase colados, porque com algumas dezenas de features e dezenas de milhares de linhas este
modelo não tem capacidade para decorar. À **direita** (alpha alto) os coeficientes são espremidos a
quase zero e o modelo vira o chute da média: as duas curvas sobem juntas. É **underfitting**, não
overfitting.

Se a curva for quase plana numa faixa larga, **prefira o alpha maior dentro do empate**: mesma
precisão com coeficientes mais estáveis, que é o que a seção 8.3 pedia.

### 10.2 · Busca aleatória no boosting

Para o modelo de árvores não dá para desenhar uma curva: são vários hiperparâmetros ao mesmo tempo.
Usamos **`RandomizedSearchCV`**, que sorteia `N_ITER` combinações, em vez de `GridSearchCV`, que
testaria o produto cartesiano inteiro.

**Por que aleatória e não grade:** com 5 hiperparâmetros e 4 valores cada, a grade são 1.024
ajustes. A busca aleatória costuma achar uma combinação equivalente numa fração disso, porque na
prática só dois ou três hiperparâmetros importam de verdade — e a grade gasta o mesmo esforço nos que
não importam.

⚠️ A busca usa **CV de 3 partes internamente** (mais barata). O resultado da busca **não** é
comparável com a tabela: por isso o vencedor é reavaliado nas mesmas 5 rodadas no fim da célula.

⚠️ **Esse número ainda é otimista.** A combinação foi escolhida olhando estas mesmas linhas, e a maior
média entre várias tende a ser a que oscilou para cima (o `best_score_` promete mais do que entrega,
a20). O número honesto do modelo escolhido é o do **teste**, na seção 13.

In [ ]:
espaco = {
    "modelo__max_iter":          [150, 300, 500],
    "modelo__learning_rate":     [0.03, 0.05, 0.1, 0.2],
    "modelo__max_leaf_nodes":    [15, 31, 63],
    "modelo__min_samples_leaf":  [20, 50, 100],
    "modelo__l2_regularization": [0.0, 0.1, 1.0],
}

busca = RandomizedSearchCV(pipe(HistGradientBoostingRegressor(random_state=SEED)),
                           param_distributions=espaco, n_iter=N_ITER, cv=CV3,
                           scoring="neg_mean_absolute_error",
                           random_state=SEED, n_jobs=-1, refit=True)
t0 = time.time()
busca.fit(X, y)
print(f"{N_ITER} combinações em {time.time() - t0:.0f}s\n")
print("melhores hiperparâmetros:")
for k, v in busca.best_params_.items():
    print(f"  {k.replace('modelo__', ''):22} {v}")

ranking = (pd.DataFrame(busca.cv_results_)
           .nsmallest(5, "rank_test_score")[["rank_test_score", "mean_test_score", "std_test_score"]]
           .assign(MAE=lambda d: -d["mean_test_score"])
           .rename(columns={"rank_test_score": "posição", "std_test_score": "dp"})
           [["posição", "MAE", "dp"]])
display(ranking.round(3))

# reavalia o vencedor nas MESMAS 5 rodadas dos outros
HIPER = {k.replace("modelo__", ""): v for k, v in busca.best_params_.items()}
res.append(avaliar("Boosting ajustado", busca.best_estimator_))
tabela(res)

**Como ler o ranking:** compare o MAE do 1º com o do 5º. Se a diferença couber dentro do `dp`, o
tuning **não achou nada** — os hiperparâmetros não são o que limita este problema, e o grupo pode
dizer isso com números em vez de insistir em buscas maiores. É um resultado legítimo e, dado o teto
previsto na seção 2.10, o desfecho mais provável.

## 11 · Ensemble

Random Forest e boosting já são ensembles, mas **de árvores com árvores**. Combinar modelos que erram
do **mesmo jeito** não ajuda; o ganho vem de juntar famílias que erram de forma **diferente**.

- **Voting** — média simples das previsões. Sem parâmetro para ajustar, sem risco de vazamento.
- **Stacking** — treina um **meta-modelo** (aqui, um `RidgeCV`) para aprender **quanto pesar** cada
  modelo base. Mais poderoso e mais caro: precisa de validação cruzada interna para gerar as
  previsões de treino do meta-modelo, senão vaza.

⏱️ O Stacking reajusta todos os modelos base várias vezes. É de longe a célula mais lenta.

In [ ]:
bases = [("ridge",    pipe(Ridge(alpha=melhor_alpha, random_state=SEED))),
         ("floresta", pipe(FLORESTA)),
         ("boosting", pipe(HistGradientBoostingRegressor(random_state=SEED, **HIPER)))]

votacao = VotingRegressor(bases, n_jobs=-1)
empilha = StackingRegressor(bases, final_estimator=RidgeCV(), cv=CV3, n_jobs=-1)

print("rodando os ensembles (lento):")
res.append(avaliar("Ensemble: Voting (média)", votacao))
res.append(avaliar("Ensemble: Stacking (RidgeCV)", empilha))
tabela(res)

In [ ]:
empilha.fit(X, y)
pesos = pd.Series(empilha.final_estimator_.coef_, index=[n for n, _ in bases])
print("peso que o meta-modelo deu a cada modelo base:")
print(pesos.round(3).to_string())
print(f"\nintercepto: {empilha.final_estimator_.intercept_:.1f}")

**Como ler:** se o ensemble empata com o melhor modelo isolado, a conclusão é que **os modelos estão
errando nas mesmas linhas** — não há diversidade para explorar. Num problema em que o teto é dado
pelo ruído do alvo, isso é o esperado.

Se o Stacking deu quase todo o peso a um modelo só, ele **descobriu sozinho** que os outros não
acrescentam.

## 12 · Interpretabilidade

Duas ferramentas, e elas respondem a perguntas **diferentes**:

| | Pergunta | Custo |
|---|---|---|
| **Permutation importance** | se eu **embaralhar** esta coluna, quanto o modelo piora? | barato, funciona em qualquer modelo |
| **SHAP** | para **esta pessoa**, quantos pontos cada feature somou ou tirou da previsão? | caro, mas dá o sinal e o caso individual |

A permutation importance é medida num pedaço **separado** (`X_val`), porque importância medida no
próprio treino premia quem decorou. As duas explicam o **boosting ajustado** da seção 10.2; se a
seção 12.1 ficar com outro modelo, troque `modelo_interp` por ele.

Como os índices (`indice_bens`, `inse_proxy`) são calculados dentro do pipeline, a importância é
medida nas colunas de **entrada**: embaralhar `log_renda_pc` também mexe no `inse_proxy` que sai dela.

⚠️ **Cuidado com features correlacionadas.** Embaralhar `log_renda_pc` sozinho não derruba muito o
modelo se `inse_proxy` e `renda_sm` continuam lá dizendo quase a mesma coisa. A importância aparece
**dividida** entre colunas redundantes. É a multicolinearidade da seção 8.3 aparecendo de outro jeito.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.25, random_state=SEED)

# o modelo que vamos explicar: o boosting ajustado da seção 10.2
modelo_interp = pipe(HistGradientBoostingRegressor(random_state=SEED, **HIPER))
modelo_interp.fit(X_tr, y_tr)
prev_val = modelo_interp.predict(X_val)
print(f"na validação separada: MAE {np.abs(y_val - prev_val).mean():.1f} pontos | "
      f"R² {modelo_interp.score(X_val, y_val):.3f}")

imp = permutation_importance(modelo_interp, X_val, y_val, n_repeats=5,
                             random_state=SEED, n_jobs=-1, scoring="neg_mean_absolute_error")
ordem = imp.importances_mean.argsort()[::-1][:15]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh([X.columns[i] for i in ordem][::-1], imp.importances_mean[ordem][::-1],
        xerr=imp.importances_std[ordem][::-1], color=AZUL, error_kw=dict(ecolor=CINZA, lw=1))
ax.set_xlabel("aumento do MAE ao embaralhar a coluna (pontos)")
ax.set_title("Permutation importance (15 maiores, medida fora do treino)")
plt.tight_layout(); plt.show()

### SHAP

`shap` não vem no Colab por padrão. Se a célula abaixo avisar que falta, rode `!pip install shap` e
repita — e, se não quiser a dependência, a permutation importance acima **já cumpre** o item de
interpretabilidade da checklist.

Usamos o `TreeExplainer` sobre o **mesmo boosting ajustado** da permutation importance — o SHAP tem de
explicar o modelo que está sendo avaliado, não outro — e sobre uma **amostra pequena**: o custo do
SHAP em árvores cresce rápido.

In [ ]:
try:
    import shap
except ImportError:
    shap = None
    print("shap não instalado. Rode:  !pip install shap   e execute esta célula de novo.")
    print("A permutation importance acima já cobre o item de interpretabilidade.")

if shap is not None:
    N_SHAP = 1_000 if RAPIDO else 3_000
    Xt = modelo_interp.named_steps["colunas"].transform(modelo_interp.named_steps["preparo"].transform(
        X_val.sample(min(N_SHAP, len(X_val)), random_state=SEED)))
    Xt = Xt.toarray() if hasattr(Xt, "toarray") else Xt
    nomes = [n.split("__", 1)[-1] for n in modelo_interp.named_steps["colunas"].get_feature_names_out()]

    t0 = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        valores = shap.TreeExplainer(modelo_interp.named_steps["modelo"]).shap_values(Xt)
    print(f"SHAP em {len(Xt):,} linhas: {time.time() - t0:.0f}s")

    shap.summary_plot(valores, Xt, feature_names=nomes, max_display=15, show=False)
    plt.title("SHAP: efeito de cada feature na nota prevista", fontsize=11)
    plt.tight_layout(); plt.show()

**Como ler o summary plot:** cada ponto é **uma pessoa**. A posição horizontal é quantos **pontos de
nota** aquela feature somou (direita) ou tirou (esquerda) da previsão dela. A cor é o **valor** da
feature — vermelho alto, azul baixo.

"Vermelho à direita" em `log_renda_pc` lê-se: *renda alta empurra a nota prevista para cima*. Uma
faixa **larga** quer dizer que a feature pesa muito em algumas pessoas e quase nada em outras — algo
que nem o coeficiente da seção 8.2 nem a permutation importance mostram.

⚠️ **Continua sendo associação, não causa.** O SHAP explica **o modelo**, não o mundo.

### 12.1 · Resumo da modelagem

A célula abaixo monta o resumo a partir dos resultados que acabaram de rodar, inclusive o veredito
sobre trocar ou não a referência linear. A regra é a da seção Objetivo: escolhemos pelo **MAE**, e o ganho só conta se
**passar das barras de erro**.

In [ ]:
t = pd.DataFrame(res).set_index("modelo")
LINEAR = "Regressão linear (referência)"
melhor = t["MAE"].idxmin()                      # a métrica principal decide (seção Objetivo)

ganho  = t.loc[LINEAR, "MAE"] - t.loc[melhor, "MAE"]      # pontos de MAE a menos que a linear
barras = t.loc[melhor, "MAE dp"] + t.loc[LINEAR, "MAE dp"]
top3 = [X.columns[i] for i in imp.importances_mean.argsort()[::-1][:3]]

def linha(nome):
    return (f"MAE {t.loc[nome, 'MAE']:.2f} +/- {t.loc[nome, 'MAE dp']:.2f} | "
            f"RMSE {t.loc[nome, 'RMSE']:.2f} | R2 {t.loc[nome, 'R2']:.3f} +/- {t.loc[nome, 'R2 dp']:.3f}")

print("RESUMO DA MODELAGEM (so treino)")
print("=" * 70)
print(f"melhor em MAE     : {melhor}")
print(f"                    {linha(melhor)}")
print(f"referencia linear : {linha(LINEAR)}")
print(f"\nganho em MAE sobre a linear : {ganho:+.2f} pontos")
print(f"soma das barras de erro     : {barras:.2f} pontos")
if ganho > barras:
    print("VEREDITO: o ganho PASSA das barras de erro -> trocar pelo modelo novo se valer o custo.")
else:
    print("VEREDITO: o ganho NAO passa das barras de erro -> a escolha honesta e ficar com a linear")
    print("          (mais barata, coeficiente legivel; o teto vem do ruido do alvo, nao do modelo).")
escolhido = melhor if ganho > barras else LINEAR
print(f"modelo escolhido  : {escolhido}")
print(f"criterio 'bom o suficiente' (MAE < 113): "
      f"{'atingido' if t.loc[escolhido, 'MAE'] < 113 else 'NAO atingido'} ({t.loc[escolhido, 'MAE']:.2f})")

print(f"\nRidge             : melhor alpha {melhor_alpha:.3g}")
print(f"tuning            : {N_ITER} combinacoes; melhor MAE da busca {-busca.best_score_:.2f} (otimista)")
print(f"                    {busca.best_params_}")
print(f"ensembles         : " + " | ".join(
    f"{n.split(': ')[-1]} MAE {t.loc[n, 'MAE']:.2f}" for n in t.index if n.startswith("Ensemble")))
print(f"pesos do stacking : {pesos.round(3).to_dict()}")
print(f"top 3 features (permutation importance): {', '.join(top3)}")
print(f"custo             : melhor {t.loc[melhor, 'seg']}s vs linear {t.loc[LINEAR, 'seg']}s")


## 13 · Avaliação no conjunto de teste

**Pendente.** Rodar **uma vez só**, depois de decidido o modelo na seção 12.1.

Usa as linhas `split == "teste"` de `enem2024_features.parquet`, guardadas desde a seção 4.1. Nada
de tuning aqui: se o número do teste não agradar e o grupo voltar para a seção 10, o teste deixa de
ser teste e vira mais um conjunto de validação.

O pipeline da seção 6 é usado inteiro, sem exceção: o preparo é ajustado **só** no treino, como em
qualquer rodada da validação cruzada.

## 14 · Checklist ético e de viés

**Pendente.** Consolida o que já apareceu espalhado:

- quem está fora da base e por quê (seção 2.4: 26,9% não fizeram a redação, e a falta não é aleatória);
- o erro por grupo do modelo de referência (seção 8.5);
- a leitura de associação × causa das seções 8.2 e 12.

## 15 · Deploy (Streamlit)

**Pendente.** Empacotar o modelo escolhido e o app. Não entra no `run_all.sh`.